# Notebook 3: Generation Evaluation

Evaluate the generated answer — two ways: **semantic similarity** (no LLM needed, compares to a reference answer) and **LLM-as-judge** (no reference needed, scores faithfulness and relevance directly).

## Setup

In [1]:
import json
import math
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
client          = OpenAI()
MODEL           = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
DATA_PATH       = Path("../data/AI_Agent_Insure.md")
GOLDEN_PATH     = Path("../data/golden_dataset.json")
TOP_K           = 8

## Part 1: Build the RAG pipeline

Same pipeline as Modules 4–6. We need it to generate answers to evaluate.

In [2]:
def load_and_chunk(path: Path) -> list[str]:
    doc    = path.read_text(encoding="utf-8").strip()
    chunks = [s.strip() for s in doc.replace("\n", " ").split(".") if s.strip()]
    return [c + "." for c in chunks]


chunks     = load_and_chunk(DATA_PATH)
resp       = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks)
embeddings = [r.embedding for r in resp.data]

chroma = chromadb.EphemeralClient()
coll   = chroma.create_collection("evals_gen")
coll.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks
)
print(f"Index ready: {len(chunks)} chunks")


def retrieve(query: str, k: int = TOP_K) -> list[str]:
    q_emb   = client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    q_vec   = q_emb.data[0].embedding
    results = coll.query(query_embeddings=[q_vec], n_results=k)
    return results["documents"][0]


def generate(question: str, context_chunks: list[str]) -> str:
    context = "\n---\n".join(context_chunks)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for AI Agent Insure. "
                "Answer the question using only the provided context. "
                "If the context doesn't contain the answer, say so clearly."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0
    )
    return resp.choices[0].message.content


def rag(question: str) -> tuple[str, list[str]]:
    context_chunks = retrieve(question)
    answer         = generate(question, context_chunks)
    return answer, context_chunks

Index ready: 34 chunks


## Part 2: Method 1 — Semantic similarity (reference-based)

Compare the model's answer to the expected answer using **cosine similarity on embeddings**.

- Score = 1.0 → answers are semantically identical
- Score > 0.85 → strong agreement
- Score < 0.6 → meaningfully different

No LLM needed for scoring — just two embedding calls.

In [3]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot     = sum(x * y for x, y in zip(a, b))
    norm_a  = math.sqrt(sum(x * x for x in a))
    norm_b  = math.sqrt(sum(x * x for x in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def semantic_similarity(text_a: str, text_b: str) -> float:
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text_a, text_b])
    return cosine_similarity(resp.data[0].embedding, resp.data[1].embedding)


print("Semantic similarity function defined.")

# Quick sanity check.
score_same = semantic_similarity(
    "AI Agent Insure covers autonomous AI agents.",
    "AI Agent Insure provides coverage for autonomous AI agents."
)
score_diff = semantic_similarity(
    "AI Agent Insure covers autonomous AI agents.",
    "The weather in San Francisco is often foggy."
)
print(f"Similar sentences : {score_same:.3f}  (expect > 0.85)")
print(f"Different topics  : {score_diff:.3f}  (expect < 0.5)")

Semantic similarity function defined.
Similar sentences : 0.977  (expect > 0.85)
Different topics  : 0.068  (expect < 0.5)


In [4]:
golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

print("Running RAG + semantic similarity eval...")
print()

sim_results = []
for case in golden:
    actual_answer, _ = rag(case["question"])
    sim              = semantic_similarity(actual_answer, case["expected_answer"])

    sim_results.append({
        "id":       case["id"],
        "category": case["category"],
        "question": case["question"][:55],
        "sim":      round(sim, 3),
        "actual":   actual_answer,
        "expected": case["expected_answer"],
    })

print(f"{'ID':>4}  {'Cat':12}  {'Sim':>5}  Question")
print("-" * 80)
for r in sim_results:
    flag = "  " if r["sim"] >= 0.75 else " !"
    print(f"{r['id']:>4}  {r['category']:12}  {r['sim']:>5}{flag}  {r['question']}")

mean_sim = sum(r["sim"] for r in sim_results) / len(sim_results)
print()
print(f"Mean semantic similarity: {mean_sim:.3f}")

Running RAG + semantic similarity eval...

  ID  Cat             Sim  Question
--------------------------------------------------------------------------------
 q01  product_info  0.887    What does Agentic AI Liability Insurance cover?
 q02  pricing       0.808    How much would Model & Data Security Insurance cost for
 q03  eligibility   0.706 !  Can a healthcare company get coverage from AI Agent Ins
 q04  rag           0.861    How does AI Agent Insure handle claims?
 q05  rag           0.827    What is AI Agent Insure's underwriting philosophy?
 q06  product_info  0.668 !  What does Autonomous Systems & Robotics Coverage includ
 q07  pricing       0.749 !  How much does Agentic AI Liability Insurance cost for a
 q08  eligibility   0.836    Is a synthetic data company eligible for coverage?
 q09  multi_tool    0.702 !  I'm a robotics startup. Am I eligible and what would co
 q10  multi_tool    0.746 !  Explain the claims process and what Model & Data Securi

Mean semantic similarit

## Part 3: Inspect a low-scoring case

In [5]:
# Find the lowest-scoring case and look at what went wrong.
worst = min(sim_results, key=lambda r: r["sim"])

print(f"Lowest similarity: {worst['sim']} — [{worst['id']}] {worst['question']}")
print()
print("Expected answer:")
print(worst["expected"])
print()
print("Actual answer:")
print(worst["actual"])

Lowest similarity: 0.668 — [q06] What does Autonomous Systems & Robotics Coverage includ

Expected answer:
It covers physical damage, liability, and operational risk for robotics and autonomous vehicle deployments.

Actual answer:
The context does not provide specific details about what Autonomous Systems & Robotics Coverage includes.


## Part 4: Method 2 — LLM-as-judge (reference-free)

Ask a second LLM call to score the answer on two dimensions:
- **Faithfulness** — is the answer supported by the retrieved context? (catches hallucination)
- **Relevance** — does the answer actually address the question?

No reference answer needed. The judge only sees: question + context + answer.

We use structured output (JSON mode) so the score is always machine-readable.

In [6]:
JUDGE_PROMPT = """\
You are an evaluation assistant. Score the answer below on two dimensions.

QUESTION:
{question}

CONTEXT (what the system retrieved):
{context}

ANSWER (what the system generated):
{answer}

Score each dimension from 1 to 5:
- faithfulness: Is every claim in the answer supported by the context? (5 = fully grounded, 1 = mostly hallucinated)
- relevance: Does the answer directly address the question? (5 = fully answers it, 1 = off-topic)

Respond with JSON only:
{{"faithfulness": <1-5>, "relevance": <1-5>, "reasoning": "<one sentence>"}}
"""


def llm_judge(question: str, context_chunks: list[str], answer: str) -> dict:
    context = "\n---\n".join(context_chunks)
    prompt  = JUDGE_PROMPT.format(
        question=question, context=context, answer=answer
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(resp.choices[0].message.content)


print("LLM judge function defined.")

LLM judge function defined.


In [7]:
print("Running LLM-as-judge eval...")
print()

judge_results = []
for case in golden:
    actual_answer, context_chunks = rag(case["question"])
    scores = llm_judge(case["question"], context_chunks, actual_answer)

    judge_results.append({
        "id":          case["id"],
        "category":    case["category"],
        "question":    case["question"][:55],
        "faithfulness": scores.get("faithfulness", 0),
        "relevance":    scores.get("relevance", 0),
        "reasoning":    scores.get("reasoning", ""),
        "actual":       actual_answer,
    })

print(f"{'ID':>4}  {'Cat':12}  {'Faith':>5}  {'Rel':>5}  Question")
print("-" * 80)
for r in judge_results:
    flag = "  " if r["faithfulness"] >= 4 and r["relevance"] >= 4 else " !"
    print(f"{r['id']:>4}  {r['category']:12}  {r['faithfulness']:>5}  {r['relevance']:>5}{flag}  {r['question']}")

mean_f = sum(r["faithfulness"] for r in judge_results) / len(judge_results)
mean_r = sum(r["relevance"]    for r in judge_results) / len(judge_results)
print()
print(f"Mean faithfulness : {mean_f:.2f} / 5")
print(f"Mean relevance    : {mean_r:.2f} / 5")

Running LLM-as-judge eval...

  ID  Cat           Faith    Rel  Question
--------------------------------------------------------------------------------
 q01  product_info      3      4 !  What does Agentic AI Liability Insurance cover?
 q02  pricing           5      5    How much would Model & Data Security Insurance cost for
 q03  eligibility       5      4    Can a healthcare company get coverage from AI Agent Ins
 q04  rag               5      5    How does AI Agent Insure handle claims?
 q05  rag               5      5    What is AI Agent Insure's underwriting philosophy?
 q06  product_info      3      2 !  What does Autonomous Systems & Robotics Coverage includ
 q07  pricing           5      5    How much does Agentic AI Liability Insurance cost for a
 q08  eligibility       5      5    Is a synthetic data company eligible for coverage?
 q09  multi_tool        5      5    I'm a robotics startup. Am I eligible and what would co
 q10  multi_tool        5      3 !  Explain the clai

## Part 5: Inspect the judge's reasoning

In [8]:
# Print the judge's reasoning for every case to understand what it's measuring.
for r in judge_results:
    print(f"[{r['id']}] F={r['faithfulness']} R={r['relevance']}")
    print(f"  Q: {r['question']}")
    print(f"  Judge: {r['reasoning']}")
    print()

[q01] F=3 R=4
  Q: What does Agentic AI Liability Insurance cover?
  Judge: The answer correctly identifies that Agentic AI Liability Insurance covers risks associated with agentic AI systems, but it lacks specific details about the types of risks covered as mentioned in the context.

[q02] F=5 R=5
  Q: How much would Model & Data Security Insurance cost for
  Judge: The answer accurately reflects the context by stating that there is no information about the cost of Model & Data Security Insurance for a startup.

[q03] F=5 R=4
  Q: Can a healthcare company get coverage from AI Agent Ins
  Judge: The answer accurately reflects the lack of specific information in the context regarding coverage for healthcare companies, but it could have elaborated on the types of risks covered that might apply to healthcare.

[q04] F=5 R=5
  Q: How does AI Agent Insure handle claims?
  Judge: The answer accurately reflects the context by stating that claims are handled through integrated claims and incid

## Part 6: LLM-as-judge failure modes

LLM-as-judge is powerful but has its own failure modes you need to know about:

| Failure mode | What happens | Mitigation |
|---|---|---|
| **Sycophancy** | Judge gives high scores to confident-sounding answers even if wrong | Use a stricter prompt; ask for evidence |
| **Position bias** | Judge favors answers that appear first in the prompt | Randomise order; average two calls |
| **Verbosity bias** | Judge rewards longer answers regardless of quality | Explicitly penalise unnecessary length in the prompt |
| **Self-consistency** | Same model judging its own outputs inflates scores | Use a different (ideally stronger) model as judge |

In [9]:
# Demo: show that a hallucinated answer gets a low faithfulness score.
hallucinated_answer = (
    "AI Agent Insure offers a 50% discount for startups in their first year, "
    "and all policies include free legal representation in any jurisdiction worldwide."
)

sample_case    = golden[0]
_, ctx_chunks  = rag(sample_case["question"])
scores         = llm_judge(sample_case["question"], ctx_chunks, hallucinated_answer)

print(f"Question  : {sample_case['question']}")
print(f"Answer    : {hallucinated_answer}")
print(f"Scores    : faithfulness={scores['faithfulness']}, relevance={scores['relevance']}")
print(f"Reasoning : {scores['reasoning']}")

Question  : What does Agentic AI Liability Insurance cover?
Answer    : AI Agent Insure offers a 50% discount for startups in their first year, and all policies include free legal representation in any jurisdiction worldwide.
Scores    : faithfulness=1, relevance=2
Reasoning : The answer provides information about discounts and legal representation that is not mentioned in the context, failing to address what Agentic AI Liability Insurance specifically covers.


## Key concepts

- **Semantic similarity** is fast and cheap but requires a reference answer — good for regression testing
- **LLM-as-judge** requires no reference but costs an extra API call per evaluation — good for production monitoring
- Use both: similarity to catch regressions, LLM-as-judge to understand *why* something regressed
- LLM-as-judge has its own failure modes — prompt it defensively
- **Next:** Notebook 4 evaluates the agent layer — tool routing and argument correctness